## CELDA 1 — Imports y carga

In [ ]:
# ─── Preprocesamiento · p7_g1_multiclase ──────────────────────────────────────

# pandas: manipulación de DataFrames (tablas de datos)
import pandas as pd

# numpy: operaciones matemáticas y manejo de arrays
import numpy as np

# train_test_split: divide el dataset en conjunto de entrenamiento y prueba
from sklearn.model_selection import train_test_split

# LabelEncoder:   convierte etiquetas de texto a números enteros (0, 1, 2...)
# StandardScaler: escala las features a media=0, desviación típica=1
# OrdinalEncoder: codifica variables categóricas respetando un orden explícito
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder

# Pipeline y ColumnTransformer: encadenan transformaciones (importados para
# posibles extensiones futuras del notebook)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# joblib: serializa objetos Python a disco (para guardar el scaler y el encoder)
# os:     operaciones del sistema de archivos (crear carpetas, rutas)
import joblib, os

# Cargar el CSV limpio generado por el notebook EDA (ya sin duplicados)
# shape[0] = número de filas, shape[1] = número de columnas
df = pd.read_csv('../data/processed/ObesityDataSet_clean.csv')
print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")

# Verificación: FAVC debe mostrar 'yes'/'no' — si muestra 0/1 se está leyendo
# un CSV ya transformado y hay que volver al original
print(f"   Valores únicos en 'FAVC' (debe ser yes/no): {df['FAVC'].unique()}")

# Muestra las 3 primeras filas para verificar que la carga es correcta
df.head(3)

### 🔍 Interpretación — Carga del dataset

El dataset limpio tiene **2087 filas y 17 columnas** (2111 originales menos 24 duplicados eliminados en el EDA).  
Las columnas mezclan tres tipos de variables que requieren tratamientos diferentes antes de entrenar cualquier modelo:

| Tipo | Ejemplo | Problema | Solución |
|------|---------|----------|----------|
| Binaria texto | `yes` / `no` | Los modelos no entienden texto | Mapeo directo a `1` / `0` |
| Categórica ordinal | `Sometimes`, `Frequently` | Tiene orden lógico que debe respetarse | `OrdinalEncoder` con orden explícito |
| Categórica nominal | `Gender` | Sin orden natural | One-Hot Encoding (`get_dummies`) |
| Numérica continua | `Age`, `Weight` | Distintas escalas | `StandardScaler` (solo para KNN y SVM) |

> El preprocesamiento **no modifica el número de filas** — solo transforma los valores de las columnas.

---
## CELDA 2 — Identificar columnas por tipo

In [ ]:
# ─── Clasificar columnas ───────────────────────────────────────────────────────

# Variable objetivo (lo que queremos predecir)
TARGET = 'NObeyesdad'

# Columnas con valores yes/no → se convertirán a 1/0 con un mapeo directo
# Son binarias: solo tienen dos valores posibles y no tienen orden entre sí
cols_binarias = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']

# Columnas categóricas con orden natural (ordinales)
# Se listan las categorías de menor a mayor para que OrdinalEncoder
# asigne 0=menor frecuencia/intensidad → 3=mayor frecuencia/intensidad
# MTRANS se codifica ordinalmente para evitar la explosión de columnas
# del One-Hot en una variable con 5 valores posibles
cols_ordinales = {
    'CAEC':  ['no', 'Sometimes', 'Frequently', 'Always'],   # consumo de comida entre horas
    'CALC':  ['no', 'Sometimes', 'Frequently', 'Always'],   # consumo de alcohol
    'MTRANS': ['Walking', 'Bike', 'Motorbike', 'Public_Transportation', 'Automobile'],  # transporte
}

# Columnas categóricas sin orden (nominales) → One-Hot Encoding
# Gender solo tiene 2 valores, con drop_first=True basta una columna: Gender_Male
cols_nominales = ['Gender']

# Columnas numéricas continuas (ya están en formato numérico, no necesitan encoding)
# Sí necesitarán escalado para KNN y SVM
cols_numericas = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

# Verificación: imprime qué columnas caen en cada categoría
print("Binarias:  ", cols_binarias)
print("Ordinales: ", list(cols_ordinales.keys()))
print("Nominales: ", cols_nominales)
print("Numéricas: ", cols_numericas)

### 🔍 Interpretación — Clasificación de columnas

El dataset tiene **17 columnas** que se distribuyen así:

- **1** columna target (`NObeyesdad`) → se codifica aparte con orden clínico
- **4** binarias (`yes`/`no`) → mapeo directo `1`/`0`
- **3** ordinales → `OrdinalEncoder` con orden explícito definido manualmente
- **1** nominal → `get_dummies` genera la columna `Gender_Male` (1=Male, 0=Female)
- **8** numéricas → ya son números, no necesitan encoding (sí escalado más adelante)

**Total: 1 + 4 + 3 + 1 + 8 = 17 ✅**

> ⚠️ **Por qué no usar `LabelEncoder` para las ordinales:** `LabelEncoder` ordena alfabéticamente,  
> lo que asignaría `Always=0`, `Frequently=1`, `no=2`, `Sometimes=3` — un orden sin sentido clínico.  
> `OrdinalEncoder` con `categories` explícitas garantiza que `no(0) < Sometimes(1) < Frequently(2) < Always(3)`.

---
## CELDA 3 — Encoding de binarias y target

In [ ]:
# ─── Encoding binarias (yes=1, no=0) ──────────────────────────────────────────

# Diccionario de traducción: 'yes' → 1, 'no' → 0
binary_map = {'yes': 1, 'no': 0}

# Aplicar el mapeo a cada columna binaria
# .map() sustituye cada valor por el correspondiente del diccionario
# Si encuentra un valor no definido en el dict, lo convierte en NaN
for col in cols_binarias:
    df[col] = df[col].map(binary_map)

# ─── Encoding del target con orden clínico real ───────────────────────────────
# LabelEncoder ordena alfabéticamente → NO preserva la progresión de obesidad
# Orden alfabético incorrecto: Insufficient(0) Normal(1) Obesity_I(2) Obesity_II(3)
#                               Obesity_III(4) Overweight_I(5) Overweight_II(6)
# Solución: definir el orden clínico correcto manualmente

orden_clinico = [
    'Insufficient_Weight',   # 0 — por debajo del peso normal
    'Normal_Weight',         # 1 — peso saludable
    'Overweight_Level_I',    # 2 — sobrepeso leve
    'Overweight_Level_II',   # 3 — sobrepeso moderado
    'Obesity_Type_I',        # 4 — obesidad grado I
    'Obesity_Type_II',       # 5 — obesidad grado II
    'Obesity_Type_III',      # 6 — obesidad grado III (máxima severidad)
]

# Construir el diccionario de mapeo: {'Insufficient_Weight': 0, 'Normal_Weight': 1, ...}
# enumerate(orden_clinico) genera pares (índice, valor): (0,'Insufficient_Weight'), ...
# El índice 'i' es la etiqueta numérica que se asignará a cada clase (0, 1, 2...)
target_map = {clase: i for i, clase in enumerate(orden_clinico)}

# Aplicar el mapeo al target → nueva columna numérica 'target_encoded'
df['target_encoded'] = df[TARGET].map(target_map)

# Crear el LabelEncoder con el orden clínico para poder invertir predicciones
# (número → nombre de clase) en el notebook de modelado
# Se asigna classes_ directamente en lugar de usar fit() para respetar nuestro orden
le_target = LabelEncoder()
le_target.classes_ = np.array(orden_clinico)

# ─── Verificación ─────────────────────────────────────────────────────────────
# assert lanza un error si la condición es False
# isna().sum() cuenta los NaN: debe ser 0 (ningún valor sin mapear)
assert df['target_encoded'].isna().sum() == 0, "❌ Hay valores no mapeados en target"

# nunique() cuenta los valores únicos → debe ser 7 (una por clase)
print(f"\n✅ Mapeo correcto — {df['target_encoded'].nunique()} clases, sin NaN")

# Tabla de equivalencias: muestra solo las 7 filas únicas (una por clase)
# drop_duplicates() → una fila por clase (no cuenta registros, solo muestra el mapeo)
# sort_values()     → ordena por código numérico (0→6)
# reset_index(drop=True) → reinicia el índice para que muestre 0-6 limpiamente
print(df[[TARGET, 'target_encoded']]
      .drop_duplicates()
      .sort_values('target_encoded')
      .reset_index(drop=True))

# Conteo de registros por clase: value_counts() cuenta cuántas filas hay de cada clase
# El total debe ser 2087
print("\n📊 Registros por clase:")
conteo = df[TARGET].value_counts()
total = 0
for i, cls in enumerate(orden_clinico):
    n = conteo[cls]
    total += n
    print(f"  {i} → {cls:<25} {n:>4} registros")
print(f"  {'TOTAL':<26} {total:>4} registros")

### 🔍 Interpretación — Encoding de binarias y target

**Variables binarias:** Las 4 columnas `yes`/`no` se convierten a `1`/`0`. Es necesario porque los algoritmos de ML operan con números — no pueden calcular distancias ni productos escalares con texto.

**Target con orden clínico:** La codificación `0→6` refleja la progresión real de la enfermedad:

```
Insufficient(0) → Normal(1) → Overweight_I(2) → Overweight_II(3)
                            → Obesity_I(4) → Obesity_II(5) → Obesity_III(6)
```

Esto tiene dos ventajas concretas:
1. La **matriz de confusión** queda ordenada: los errores del modelo aparecen cerca de la diagonal (confundir Obesity_I con Obesity_II es esperable; confundirla con Insufficient_Weight sería una anomalía grave).
2. Si en el futuro se plantea como **regresión ordinal**, el orden ya está codificado correctamente.

**Sobre el conteo final:** La suma de registros por clase debe ser exactamente **2087**. Si no cuadra, hay un problema en la carga del CSV de entrada.

---
## CELDA 3b — Verificación visual del DataFrame tras encoding parcial

In [ ]:
# Muestra las primeras filas para confirmar visualmente que:
# · Las columnas binarias muestran 0/1 (no yes/no)
# · La columna target_encoded muestra valores entre 0 y 6
# · Gender, CAEC, CALC y MTRANS siguen siendo texto (aún no se han transformado)
df.head()

### 🔍 Interpretación — Estado intermedio del DataFrame

En este punto el DataFrame está **transformado parcialmente**:
- ✅ Columnas binarias: ya son `0`/`1`
- ✅ `target_encoded`: ya es numérico (`0`–`6`)
- ⏳ `Gender`, `CAEC`, `CALC`, `MTRANS`: aún son texto — se transforman en la siguiente celda

El número de filas sigue siendo **2087** — el encoding no elimina ni añade filas, solo cambia los valores de las celdas.

---
## CELDA 4 — Encoding ordinales y nominales

In [ ]:
# ─── Ordinales ────────────────────────────────────────────────────────────────

# Recorre el diccionario cols_ordinales: {nombre_columna: [orden_de_categorias]}
for col, orden in cols_ordinales.items():
    # OrdinalEncoder convierte categorías de texto a números respetando el orden
    # categories=[orden] le indica explícitamente cuál es el orden correcto:
    # Ej: ['no','Sometimes','Frequently','Always'] → asigna [0.0, 1.0, 2.0, 3.0]
    oe = OrdinalEncoder(categories=[orden])

    # fit_transform aprende el mapeo Y lo aplica en un solo paso
    # df[[col]] (doble corchete) pasa un DataFrame de una columna en lugar de una Serie,
    # que es el formato que espera OrdinalEncoder
    df[col] = oe.fit_transform(df[[col]])

# ─── Nominales (Gender) → One-Hot ─────────────────────────────────────────────

# get_dummies crea una columna binaria por cada categoría única
# drop_first=True elimina la primera columna generada para evitar multicolinealidad:
# con Gender (Male/Female) genera solo Gender_Male (1=Male, 0=Female)
# La columna 'Gender' original desaparece y es reemplazada por 'Gender_Male'
df = pd.get_dummies(df, columns=cols_nominales, drop_first=True)

# Verificación: imprime el tipo de dato de cada columna
# Todas deben ser numéricas (int64, float64 o bool/uint8) — ninguna debe ser 'object'
print("✅ Encoding completado")
print(df.dtypes)

### 🔍 Interpretación — Encoding ordinales y nominales

Tras esta celda el DataFrame es **100% numérico** — no queda ninguna columna de tipo `object` (texto).

**Resultado del encoding ordinal:**

| Columna | Valor original | Valor codificado |
|---------|---------------|------------------|
| `CAEC` | `'Sometimes'` | `1.0` |
| `CALC` | `'no'` | `0.0` |
| `MTRANS` | `'Public_Transportation'` | `3.0` |

**Resultado del One-Hot en Gender:**
- La columna `Gender` desaparece del DataFrame
- Aparece `Gender_Male` (tipo `bool` o `uint8`): `True`/`1` = Male, `False`/`0` = Female
- El número de columnas **no aumenta** — Gender se reemplaza por Gender_Male

> `drop_first=True` es importante: sin él, `Gender_Male` y `Gender_Female` serían perfectamente redundantes  
> (una es el complemento exacto de la otra), lo que introduce **multicolinealidad** y puede desestabilizar modelos lineales.

---
## CELDA 5 — Split train/test

In [ ]:
# ─── Split estratificado (mantiene proporción de clases) ──────────────────────

# X: todas las columnas EXCEPTO el target original y su versión codificada
# Son las features (variables de entrada) que el modelo usará para predecir
X = df.drop(columns=[TARGET, 'target_encoded'])

# y: solo la columna target codificada (valores 0-6)
# Es la variable de salida que el modelo intentará predecir
y = df['target_encoded']

# Dividir en train (80%) y test (20%)
# test_size=0.2   → el 20% de los datos van a test (~418 filas)
# random_state=42 → semilla fija para reproducibilidad (siempre el mismo split)
# stratify=y      → garantiza que cada clase tenga la misma proporción en
#                   train y test (crítico en multiclase para evitar que alguna
#                   clase quede infrarrepresentada o ausente en test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")

# value_counts() cuenta cuántos registros hay de cada clase (0-6)
# sort_index() ordena por código numérico para que salga en orden clínico
# Los valores deben ser proporcionales entre train y test
print(f"\nDistribución y_train:\n{y_train.value_counts().sort_index()}")
print(f"\nDistribución y_test:\n{y_test.value_counts().sort_index()}")

### 🔍 Interpretación — Split train/test

Con 2087 filas y `test_size=0.2`:
- **Train:** 1669 filas (80%) — el modelo **aprende** de estos datos
- **Test:** 418 filas (20%) — se usan **solo para evaluar**, nunca para entrenar

El parámetro `stratify=y` es fundamental en clasificación multiclase. Sin él, el split aleatorio podría dejar alguna clase con muy pocos ejemplos en test (o incluso ninguno), haciendo la evaluación poco fiable. Con `stratify`, cada clase mantiene su proporción original en ambos conjuntos.

**Ejemplo con Obesity_Type_I (351 registros = 16.8% del total):**
- En train: ~281 registros (~16.8% de 1669)
- En test: ~70 registros (~16.8% de 418)

> ⚠️ **Regla de oro del ML:** Una vez hecho el split, el conjunto test **no se toca** hasta la evaluación final.  
> Todo el preprocesamiento posterior (escalado) se aprende **solo con train** y luego se aplica a test.

---
## CELDA 6 — Escalado (solo para KNN y SVM)

In [ ]:
# ─── Scaler FIT solo sobre train, transform sobre ambos ───────────────────────
# ⚠️  Árboles y XGBoost NO necesitan escalado → usarán X_train / X_test directos
# KNN y SVM sí necesitan escalado → usarán X_train_scaled / X_test_scaled

# StandardScaler transforma cada columna para que tenga media=0 y std=1
# Fórmula aplicada a cada valor: z = (x - media_columna) / std_columna
scaler = StandardScaler()

# fit_transform sobre X_train:
# · fit()      → calcula la media y std de CADA columna usando solo datos de train
# · transform() → aplica z = (x - media) / std a todos los valores
# Se envuelve en DataFrame para conservar nombres de columnas e índice original
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# transform (SIN fit) sobre X_test:
# Aplica la media y std calculadas en train — NO recalcula nada con los datos de test
# ⛔ NUNCA usar fit_transform aquí: recalcular con test haría que el modelo vea
#    información del conjunto de evaluación durante el entrenamiento (data leakage),
#    lo que inflaría artificialmente las métricas finales
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("✅ Escalado completado")

# describe() muestra estadísticas básicas por columna
# En X_train_scaled las columnas numéricas deben tener mean ≈ 0.00 y std ≈ 1.00
# Las columnas binarias (0/1) también se escalan: sus medias serán distintas de 0
print(X_train_scaled.describe().round(2))

### 🔍 Interpretación — Escalado

**¿Por qué escalar KNN y SVM?**  
KNN calcula distancias euclidianas entre puntos. Sin escalado, `Weight` (rango 40–170 kg) domina completamente sobre `FAF` (rango 0–3), haciendo que el modelo ignore las variables de escala pequeña. SVM tiene el mismo problema al maximizar el margen entre clases.

**¿Por qué NO escalan los árboles?**  
Decision Tree, Random Forest y XGBoost hacen splits binarios del tipo `Weight > 85?`. El umbral de corte se adapta automáticamente a la escala — escalar no cambia el orden relativo de los valores ni el resultado de los splits.

**Lo que debe mostrar el `describe()` de `X_train_scaled`:**
- Columnas numéricas (`Age`, `Weight`, etc.): `mean ≈ 0.00`, `std ≈ 1.00`
- Columnas binarias (`FAVC`, `SMOKE`, etc.): `mean` será la proporción de 1s en train (ej: 0.23 si el 23% fuma)

**Qué datos usa cada modelo en el notebook de modelado:**

| Modelo | Features de entrada |
|--------|--------------------|
| Decision Tree | `X_train` / `X_test` |
| Random Forest | `X_train` / `X_test` |
| XGBoost | `X_train` / `X_test` |
| KNN | `X_train_scaled` / `X_test_scaled` |
| SVM | `X_train_scaled` / `X_test_scaled` |

---
## CELDA 7 — Guardar artefactos

In [ ]:
# ─── Guardar todo lo necesario para el notebook de modelado ───────────────────

# exist_ok=True evita error si la carpeta ya existe
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# ─── CSVs de datos ─────────────────────────────────────────────────────────────
# index=False evita que pandas guarde el índice del DataFrame como columna extra

# Para árboles, Random Forest y XGBoost (sin escalar)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)

# Para KNN y SVM (escalados con StandardScaler)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv',   index=False)

# Etiquetas (mismas para todos los modelos — valores 0-6 del target)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

# ─── Objetos sklearn serializados con joblib ───────────────────────────────────
# joblib.dump guarda el objeto Python en un fichero .pkl (pickle binario)
# scaler:    necesario para aplicar la misma transformación a nuevos datos
# le_target: necesario para convertir predicciones numéricas (0-6) a nombres de clase
joblib.dump(scaler,    '../models/scaler.pkl')
joblib.dump(le_target, '../models/label_encoder_target.pkl')

print("✅ Artefactos guardados:")
print("   · X_train / X_test              → para árboles y XGBoost")
print("   · X_train_scaled / X_test_scaled → para KNN y SVM")
print("   · y_train / y_test              → etiquetas (compartidas por todos los modelos)")
print("   · scaler.pkl                    → para escalar nuevos datos en producción")
print("   · label_encoder_target.pkl      → para invertir predicciones (número → clase)")

### 🔍 Interpretación — Artefactos guardados

Al finalizar esta celda, la carpeta del proyecto queda así:

```
data/
  processed/
    ObesityDataSet_clean.csv      ← input de este notebook (generado en EDA)
    X_train.csv                   ← features train sin escalar  (~1669 filas × 16 cols)
    X_test.csv                    ← features test  sin escalar  (~418  filas × 16 cols)
    X_train_scaled.csv            ← features train escaladas
    X_test_scaled.csv             ← features test  escaladas
    y_train.csv                   ← target train  (~1669 valores 0-6)
    y_test.csv                    ← target test   (~418  valores 0-6)
models/
    scaler.pkl                    ← StandardScaler entrenado solo con X_train
    label_encoder_target.pkl      ← LabelEncoder con orden clínico 0-6
```

**¿Por qué guardar el `scaler.pkl`?**  
Si en el futuro se quiere predecir sobre nuevos datos, hay que aplicar **exactamente la misma transformación** que se usó durante el entrenamiento (misma media y std de cada columna). Si se recalculara el scaler con los nuevos datos, los valores escalados serían distintos y el modelo daría predicciones incorrectas.

**¿Por qué guardar el `label_encoder_target.pkl`?**  
El modelo predice números (ej: `4`). Para presentar el resultado de forma comprensible hay que traducirlo a su nombre clínico (`Obesity_Type_I`). La llamada `le_target.inverse_transform([4])` hace esa conversión usando el orden clínico definido en este notebook.